#### 패키지 인스톨

In [1]:
!pip install -U transformers tqdm optuna transformers datasets accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 54.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.1
    Uninstalling transformers-4.56.1:
      Successfully uninstalled transformers-4.56.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependenc

#### 라이브러리 임포트

In [2]:
import torch
import json
import os
torch.manual_seed(123)  # 시드설정

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### 모델 불러오기

In [5]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

# Qwen3 1.7B 모델 불러오기
model_name = "Qwen/Qwen3-1.7B"


tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")  # 토크나이저 설정
model = AutoModelForCausalLM.from_pretrained(model_name)                    # 모델 설정
model.to(device)        # 모델 GPU 올리기

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwe

#### 테스트셋 제작

In [6]:
qna_list = []
max_data_set = 100

# 파일 이름을 .jsonl로 변경하고, 인코딩을 utf-8로 지정
with open("/content/drive/MyDrive/main/Qwen 학습/train_data.jsonl", "r", encoding="utf-8") as file:
    for i, line in enumerate(file):

        # 1. JSONL 파일의 각 줄을 파싱하여 파이썬 딕셔너리로 변환
        data = json.loads(line)

        # 2. 파싱된 데이터에서 질문(question)과 답변(answers) 추출
        question = data['question']
        answer = data['answer']

        # 10개만 형식을 유지하는지 확인
        if i <= 10:
            print(f'{question}\n{answer}')
            print('-'*80)

        # 3. messages 리스트 구성 // 나름의 프롬프트
        messages = [
            {"role": "system", "content": "You are an assistant that explains terms about ship building."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]


        # 질문과 전체 응답 프롬프트
        q = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
        input_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

        # 글자를 숫자로 인코딩
        q_ids = tokenizer.encode(q, add_special_tokens=False)
        input_ids = tokenizer.encode(input_str, add_special_tokens=False)

        # 딕셔너리 형태로 qna 리스트에 추가
        qna_list.append({'q': q, 'input': input_str, 'q_ids': q_ids, 'input_ids': input_ids})



# 가장 긴 토큰의 길이 확인
if qna_list: # 리스트가 비어있지 않은 경우에만 max_length 계산
    max_length = max(len(i['input_ids']) for i in qna_list)
    print(f"가장 긴 토큰의 길이: {max_length}")
else:
    print("처리된 데이터가 없습니다.")


The main agenda of the meeting was how to reduce the cost of the Service Air Reservoir.
회의의 주요 안건은 일반용공기탱크의 비용 절감 방안이었습니다.
--------------------------------------------------------------------------------
According to the report, a problem occurred in the DECIBEL section.
보고서에 따르면, 데시벨(소음 측정단위) 부분에서 문제가 발생했습니다.
--------------------------------------------------------------------------------
The system must be organically linked with the Shore Connection Box.
해당 시스템은 선외급전상자와 유기적으로 연동되어야 합니다.
--------------------------------------------------------------------------------
The standards for Slip-on Welding Flange must be strictly followed when designing a ship.
선박 설계 시 삽입용접플랜지의 기준을 반드시 준수해야 합니다.
--------------------------------------------------------------------------------
It is important to understand the concept of Tuna Long Liner in this project.
이번 프로젝트에서는 다랑어주낙어선 개념을 이해하는 것이 중요합니다.
--------------------------------------------------------------------------------
Please look for infor

#### Train, Test 데이터 셋 분리

In [7]:
from torch.utils.data import Dataset, DataLoader

# 151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
# End Of token
EOT = 151643

# 나의 데이터셋을 정의하는 클래스 객체 생성 // Dataset 상속
class MyDataset(Dataset):

    # 초기화 함수
    def __init__(self, qna_list, max_length):
        self.input_ids = []
        self.target_ids = []

        # 질문 리스트의 요소별로 정의
        for qa in qna_list:
            token_ids = qa['input_ids']     # Q&A의 전체 문자열 토큰 ID 리스트
            input_chunk = token_ids         # 이를 input_chunk
            target_chunk = token_ids[1:]    # A가 되는 target_chunck 문자열 토큰 ID 리스트

            input_chunk += [EOT]* (max_length - len(input_chunk))       # input_chunk에 [EOT]토큰 수를 최대길이에서 부족한 만큼 추가
            target_chunk +=  [EOT]* (max_length - len(target_chunk))    # target_chunk도 마찮가지로 [EOT]토큰 수를 부족한 만큼 추가

            len_ignore = len(qa['q_ids']) - 1                   # target은 한 글자가 짧기 때문
            target_chunk[:len_ignore] = [-100] * len_ignore     # 질문에 대해서는 학습하지 않도록

            self.input_ids.append(torch.tensor(input_chunk))    # input_ids에 input_chunck 추가
            self.target_ids.append(torch.tensor(target_chunk))  # target_ids에 target_chunck 추가


    # 총 데이터의 갯수를 알려주는 역할
    def __len__(self):
        return len(self.input_ids)

    # 특정 데이터를 꺼내오는 역할
    def __getitem__(self, idx):
        item = {
            'input_ids': self.input_ids[idx].clone().detach().to(torch.long), # .to(torch.long) 추가
            'labels': self.target_ids[idx].clone().detach().to(torch.long)    # .to(torch.long) 추가
        }
        return item


In [8]:
from torch.utils.data import random_split

# 데이터셋 정의
dataset = MyDataset(qna_list, max_length=max_length)

# 1. 전체 데이터셋의 크기를 계산합니다.
total_size = len(dataset)

# 2. 분리할 크기를 결정합니다 (예: 80% 학습, 20% 테스트).
train_size = 4000
test_size = 1000
unused_size = total_size - train_size - test_size

# 3. random_split 함수를 사용하여 데이터셋을 분리합니다.
train_dataset, test_dataset, _ = random_split(dataset, [train_size, test_size, unused_size])

# 4. 분리된 데이터셋의 크기를 확인합니다.
print(f"전체 데이터셋 크기: {total_size}")
print(f"학습 데이터셋 크기: {len(train_dataset)}")
print(f"테스트 데이터셋 크기: {len(test_dataset)}")

전체 데이터셋 크기: 16865
학습 데이터셋 크기: 4000
테스트 데이터셋 크기: 1000


#### optuna 하이퍼파라미터 최적화

In [9]:
import optuna
from tqdm import tqdm
import bitsandbytes as bnb
from optuna.exceptions import TrialPruned # Pruned 예외 import

# objective 함수 정의
def objective(trial):

    # =========================================

    # 파라미터 범주 설정

    # Learning Rate (5x10^-5 ~ 5x10^-4)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)

    # Epoch (1 ~ 3)
    num_train_epochs = trial.suggest_int("num_train_epochs", 1, 3)

    # Batch Size (2, 4, 8)
    per_device_train_batch_size = trial.suggest_categorical("per_device_train_batch_size", [2, 4, 8])

    # Weight Decay (0.0 ~ 0.05)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.05)

    # Warm Up Ratio (0.0 ~ 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.1)

    # Scheduler Type ("linear", "cosine", "cosine_with_restarts")
    lr_scheduler_type = trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine", "cosine_with_restarts"])

    #===========================================

    # 매 시도 마다 모델 설정

    # 어떤 LLM 모델을 불러올지
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")

    # 그래디언트 체크포인팅 -> 메모리 사용량 줄이
    model.gradient_checkpointing_enable()

    # 모델과 토크나이저의 어휘 크기를 동기화 (모델이 가진 단어보다 많은 것을 넣을 때 자동 확장)
    model.resize_token_embeddings(len(tokenizer))

    # GPU로 모델 옮기기
    model.to(device)

    #===========================================

    # 학습시 사용할 train, test 데이터 로더 설정
    train_loader = DataLoader(train_dataset, batch_size=per_device_train_batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=per_device_train_batch_size)

    # 최적화 방법 선택 (AdamW8bit - Ram 사용량 더더욱 줄이)
    optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    # GradScaler 설정 -> 소숫점 조절 및 숫자 스케일 조
    scaler = GradScaler()

    #===========================================

    # 학습률을 동적으로 설정하는 학습 스케쥴러 설정

    # 가중치가 업데이트 되는 총 횟수(스텝수) 설정
    num_training_steps = num_train_epochs * len(train_loader)
    # warmpup을 진행할 횟수
    num_warmup_steps = int(num_training_steps * warmup_ratio)

    # 실제 스케쥴러 객체 생성
    lr_scheduler = get_scheduler(
        name=lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    #===========================================

    # 학습 루프 -> trian epoch 만큼
    for epoch in range(num_train_epochs):
        # 학습모드로 모델 전환
        model.train()
        # 진행률을 tqdm으로 표시
        progress_bar = tqdm(train_loader, desc=f"Trial {trial.number} Epoch {epoch+1}/{num_train_epochs}", leave=False)

        # 매 진행 횟수마다
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['labels'].to(device)

            optimizer.zero_grad()

            with autocast():
                logits = model(input_ids).logits
                loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_ids.flatten())

            scaler.scale(loss).backward()

            # --- 그래디언트 클리핑 추가 ---
            # GradScaler를 사용할 때는 먼저 unscale을 해주어야 합니다.
            scaler.unscale_(optimizer)
            # 그래디언트의 총 norm(크기)이 1.0을 넘지 않도록 잘라냅니다.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # -----------------------------

            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step() # 스케줄러 업데이트

            progress_bar.set_postfix(loss=loss.item())

    #===========================================

    # --- ✨ 프루닝(가지치기) 로직 시작 ✨ ---
    # 1. 매 에폭마다 평가를 수행합니다.
    model.eval()
    total_eval_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['labels'].to(device)

            with autocast():
                logits = model(input_ids).logits
                loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_ids.flatten())
            total_eval_loss += loss.item()

    avg_eval_loss = total_eval_loss / len(test_loader)

    # 2. Optuna에 중간 결과를 보고합니다. (현재 에폭의 검증 손실)
    trial.report(avg_eval_loss, epoch)

    # 3. Pruning(가지치기)이 필요한지 확인합니다.
    if trial.should_prune():
        # 유망하지 않은 Trial을 조기 종료합니다.
        raise TrialPruned()
    # --- ✨ 프루닝(가지치기) 로직 끝 ✨ ---

    return avg_eval_loss

In [10]:
from optuna.pruners import MedianPruner
from torch.cuda.amp import autocast, GradScaler
from transformers import get_scheduler

# study 객체 생성 -> loss를 최소화 하는 전략으로
study = optuna.create_study(direction="minimize", pruner=MedianPruner())

# n_trials 횟수만큼 반복 진행
# 일단 10회 진행 후 시간이 된다면 나머지 10회도 진행
study.optimize(objective, n_trials=10)

# best_params 저장
best_params = study.best_params

# --- 결과 확인 ---
print("최적화 종료!")
print("최고 점수 (loss):", study.best_trial.value)
print("최적 하이퍼파라미터:", study.best_params)

[I 2025-10-01 07:47:13,978] A new study created in memory with name: no-name-d0982af3-8d42-4e99-997c-d3918b991750


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipython-input-1085390014.py:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Trial 0 Epoch 1/2:   0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipython-input-1085390014.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/tmp/ipython-input-1085390014.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[I 2025-10-01 07:58:58,263] Trial 0 finished with value: 0.10262003906816244 and parameters: {'learning_rate': 2.3174257328108297e-05, 'num_train_epochs': 2, 'per_device_train_batch_size': 4, 'weight_decay': 0.046559184044000196, 'warmup_ratio': 0.08973157037442354, 'lr_scheduler_type': 'cosine'}. Best i

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[I 2025-10-01 08:33:40,580] Trial 1 finished with value: 0.12031116212147754 and parameters: {'learning_rate': 1.1289677654230446e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 2, 'weight_decay': 0.030274447460169843, 'warmup_ratio': 0.059897394579440745, 'lr_scheduler_type': 'cosine'}. Best is trial 0 with value: 0.10262003906816244.


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[I 2025-10-01 09:28:33,753] Trial 8 finished with value: 0.09648424869775772 and parameters: {'learning_rate': 1.502956432521424e-05, 'num_train_epochs': 2, 'per_device_train_batch_size': 8, 'weight_decay': 0.001095534981222246, 'warmup_ratio': 0.09468118198388631, 'lr_scheduler_type': 'cosine_with_restarts'}. Best is trial 7 with value: 0.09597801646590233.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[I 2025-10-01 09:51:27,505] Trial 9 pruned. 


최적화 종료!
최고 점수 (loss): 0.09597801646590233
최적 하이퍼파라미터: {'learning_rate': 1.6785114767103908e-05, 'num_train_epochs': 2, 'per_device_train_batch_size': 8, 'weight_decay': 0.01507083712209732, 'warmup_ratio': 0.09580697981215358, 'lr_scheduler_type': 'linear'}


In [11]:
# -------------------------------------------------------------------
# ## 최종 모델 학습 및 저장
# -------------------------------------------------------------------

print("\n--- 최적의 하이퍼파라미터로 최종 모델 학습 시작 ---")

# 1. 최종 학습을 위한 모델을 깨끗한 상태로 다시 불러옵니다.
final_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B")
final_model.resize_token_embeddings(len(tokenizer)) # 어휘 크기 동기화
final_model.gradient_checkpointing_enable()         # 메모리 최적화
final_model.to(device)

# 2. Optuna가 찾아낸 최적의 하이퍼파라미터로 설정합니다.
final_train_loader = DataLoader(
    train_dataset,
    batch_size=best_params['per_device_train_batch_size'],
    shuffle=True
)

optimizer = bnb.optim.AdamW8bit(
    final_model.parameters(),
    lr=best_params['learning_rate'],
    weight_decay=best_params['weight_decay']
)

num_epochs = best_params['num_train_epochs']
num_training_steps = num_epochs * len(final_train_loader)
num_warmup_steps = int(num_training_steps * best_params['warmup_ratio'])

lr_scheduler = get_scheduler(
    name=best_params['lr_scheduler_type'],
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

scaler = GradScaler()

# 3. 최종 학습을 진행합니다.
for epoch in range(num_epochs):
    final_model.train()
    progress_bar = tqdm(final_train_loader, desc=f"Final Training Epoch {epoch+1}/{num_epochs}")

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['labels'].to(device)

        optimizer.zero_grad()

        with autocast():
            logits = final_model(input_ids).logits
            loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_ids.flatten())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        progress_bar.set_postfix(loss=loss.item())

# 4. 학습이 완료된 최종 모델을 저장합니다.
save_path = "/content/drive/MyDrive/main/best_model/qewn3_1dot7.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
torch.save(final_model.state_dict(), save_path)
print(f"\n✅ 최종 모델이 '{save_path}' 경로에 성공적으로 저장되었습니다.")


--- 최적의 하이퍼파라미터로 최종 모델 학습 시작 ---


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipython-input-2928872103.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Final Training Epoch 1/2:   0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipython-input-2928872103.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Final Training Epoch 2/2: 100%|██████████| 500/500 [02:53<00:00,  2.89it/s, loss=0.0553]



✅ 최종 모델이 '/content/drive/MyDrive/main/best_model/qewn3_1dot7.pth' 경로에 성공적으로 저장되었습니다.


#### 모델 학습

In [ ]:
# Ram 관리를 위한 그라디언트 체크
model.gradient_checkpointing_enable()

In [ ]:
from torch.cuda.amp import autocast, GradScaler  # 그라디언트 체크
from tqdm.auto import tqdm                       # 진행도 표시
from transformers import get_scheduler # 학습률 스케줄러를 위해 추가
from torch.utils.data import DataLoader # DataLoader 설정을 위해 추가

# --- 1. Optuna 결과 저장 ---
best_params = {
    'learning_rate': 7.060600364063313e-05,
    'num_train_epochs': 2,
    'per_device_train_batch_size': 2,
    'weight_decay': 0.029099526366805274,
    'warmup_ratio': 0.03239663807997508,
    'lr_scheduler_type': 'linear'
}

# --- 2. 최적의 배치 사이즈로 DataLoader 생성 ---
# train_dataset이 미리 정의되어 있다고 가정합니다.
train_loader = DataLoader(
    train_dataset,
    batch_size=best_params['per_device_train_batch_size'], # <-- 변경
    shuffle=True
)

# --- 3. 옵티마이저에 최적의 값 적용 ---
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_params['learning_rate'],       # <-- 변경
    weight_decay=best_params['weight_decay'] # <-- 변경
)

# --- 4. 학습률 스케줄러 생성 (새로 추가된 부분) ---
num_epochs = best_params['num_train_epochs']
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(num_training_steps * best_params['warmup_ratio'])

lr_scheduler = get_scheduler(
    name=best_params['lr_scheduler_type'],
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)
# ----------------------------------------------------

scaler = GradScaler()
tokens_seen, global_step = 0, -1
losses = []

/tmp/ipython-input-1185518069.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# --- 5. 최적의 에폭 수로 학습 반복 ---
for epoch in range(num_epochs): # <-- 변경
    model.train()
    epoch_loss = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}") # <-- 변경

    for batch in progress_bar:
        # MyDataset이 딕셔너리를 반환한다고 가정
        input_batch = batch['input_ids'].to(device)
        target_batch = batch['labels'].to(device)

        optimizer.zero_grad()

        with autocast():
            logits = model(input_batch).logits
            loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step() # <-- 스케줄러 업데이트 추가

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss}")



# 최종 모델 저장
# 'models' 라는 하위 폴더 안에 모델을 저장하고 싶을 경우
save_directory = "models"
save_path = os.path.join(save_directory, "qwen3_1dot7B_Best_Model.pth")

# (중요) 저장할 폴더가 없을 경우 오류가 발생하므로, 미리 생성해줍니다.
os.makedirs(save_directory, exist_ok=True)

# 지정된 경로에 모델 저장
torch.save(model.state_dict(), save_path)
print(f"모델이 '{save_path}' 경로에 저장되었습니다.")

Epoch 1/2:   0%|          | 0/6746 [00:00<?, ?it/s]

/tmp/ipython-input-4038132216.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 Average Loss: 0.12417504118367248


Epoch 2/2:   0%|          | 0/6746 [00:00<?, ?it/s]

Epoch 2 Average Loss: 0.04871197770802296
모델이 'models/qwen3_1dot7B_Best_Model.pth' 경로에 저장되었습니다.
